### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="labour_inspection_compliance",
    dataset_year="2019",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://doi.org/10.18710/7U6TZP",
    download_description="""
Get the data from dataverse.

wget https://dataverse.no/api/access/datafile/:persistentId?persistentId=doi:10.18710/7U6TZP/HAV8AG &&  mv  ':persistentId?persistentId=doi:10.18710%2F7U6TZP%2FHAV8AG' data.csv
mkdir -p local-data-warehouse/labour_inspection_compliance && mv data.csv local-data-warehouse/labour_inspection_compliance
""",
    # References
    academic_reference_bibtex="""@inproceedings{flogard2022dataset,
  title={A dataset for efforts towards achieving the sustainable development goal of safe working environments},
  author={Flogard, Eirik Lund and Mengshoel, Ole Jakob},
  booktitle={Thirty-sixth Conference on Neural Information Processing Systems Datasets and Benchmarks Track},
  year={2022}
}
""",
    academic_reference_bibtex_key="flogard2022dataset",
    license="CC0 1.0",
    data_tags=["IID", "Spatial", "ForcedIIDFromTemporal"],
    curation_comments="""
We start with the data from dataverse. We create a task for the Non-compliance Classification Problem (NCP).

- We resolve the industry codes to their true names.
- We drop the checklist ID, as we use the checklist text as predictive signal.
- We remove two rows with missing values for IsRegisteredVATregister, as these seem to be data artifact given the data state.
- The data has spatial information (County).
- There are lot of undescribed numeric values. Thus, we do some general purpose preprocessing and drop all columns with less than 5 not nan values
- As it turns out, the data does not contain temporal information, although it is clearly a temporal task and has data from 2021 to 2019. It seems the curators have removed the temporal information. This can mean that there is not temporal leakage or relevance for the task, or that it was forgotten. In any case, we must treat the task as IID now.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="NonCompliance",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="NonCompliance",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "data.csv", sep=";")
print("Loaded data shape:", df.shape)


# Resolve industry codes
code_resolution = pd.read_csv("klass-version-30-codes.csv", sep=";")
code_as_number = pd.to_numeric(code_resolution["code"], errors="coerce")
int_codes_mask = code_as_number.notna() & (code_as_number % 1 == 0)
code_resolution = code_resolution[int_codes_mask].copy()
code_resolution["code"] = code_as_number[int_codes_mask].astype(int)
code_resolution = code_resolution.set_index(["code", "parentCode"])["name"].to_dict()
df["Industry Sub Area"] = df.apply(
    lambda row: code_resolution[(row["Industry Code"], row["Industry Main Area Code"])],
    axis=1
)
del code_resolution

sector_map = {
    "A": "Agriculture, forestry and fishing",
    "B": "Mining and quarrying",
    "C": "Manufacturing",
    "D": "Electricity, gas, steam and air conditioning supply",
    "E": "Water supply; sewerage, waste management and remediation activities",
    "F": "Construction",
    "G": "Wholesale and retail trade; repair of motor vehicles and motorcycles",
    "H": "Transportation and storage",
    "I": "Accommodation and food service activities",
    "J": "Information and communication",
    "K": "Financial and insurance activities",
    "L": "Real estate activities",
    "M": "Professional, scientific and technical activities",
    "N": "Administrative and support service activities",
    "O": "Public administration and defence; compulsory social security",
    "P": "Education",
    "Q": "Human health and social work activities",
    "R": "Arts, entertainment and recreation",
    "S": "Other service activities",
    "T": "Activities of household as employers; undifferentiated goods- and services-producing activities of households for own account",
    "U": "Activities of extraterritorial organisations and bodies",
}
df["Industry Main Area"] = df["Industry Main Area Code"].map(sector_map)

df = df[df["IsRegisteredVATregister"].notna()]

prev_cols = set(df.columns)
df = df.dropna(axis=1, thresh=5)
print("Columns dropped due to low non-nan count:", prev_cols - set(df.columns))

df = df.drop(columns=[
    'Checklist ID',
    "Industry Main Area Code",
])
as_cat_type = [
    "Industry Code",
    "IsRegisteredVATregister",
    "IsRegisteredEmploymentregister",
    "NonCompliance",
    "Currency code",
    "Fiscal accounting type",
]
as_string_type = [
    "Checklist Content",
    "Industry Main Area",
    "Industry Sub Area",
    "County",
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicated columns
df = df.loc[:, ~df.T.duplicated()]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()